In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

#### functions

In [2]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

#### constants

In [3]:
# project
str_project = os.getcwd().split('\\')[5].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[6]
print(f'Task: {str_task}')
# output
str_dirname_output = './output'

Project: 20240423-gen-xii-payload-parsing
Task: 01_pull_local_db


#### output

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### read ssql

In [5]:
str_filepath = './sql/script.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('select \n'
 '\ttbltempstaticpool.bigAccountId,\n'
 '\ttbltempstaticpool.dtmFunded,\n'
 '\ttblDove.strRequest\n'
 'from riskdb.analytics.tbltempstaticpool\n'
 'left outer join\n'
 '(\n'
 '\tselect\n'
 '\t\tbigAccountId,\n'
 '\t\tmax(bigDoveId) as bigDoveId\n'
 '\tfrom zestdb.dbo.tblDove\n'
 '\tgroup by bigAccountId\n'
 ')tblMax1 on tblMax1.bigAccountid=tbltempstaticpool.bigAccountId\n'
 'left outer join zestdb.dbo.tblDove on tblDove.bigDoveId=tblMax1.bigDoveId\n'
 "where dtmFunded >= '01/01/2023'")


#### pull

In [6]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# rm None requests
df = df[df['strRequest'].notna()]

# show
df

<timed exec>:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


CPU times: total: 39.9 s
Wall time: 10min 2s


,bigAccountId,dtmFunded,strRequest
6690,6507952,2023-01-03,"{""request_id"":""650795220692"",""rows"":[{""row_id""..."
6691,6490808,2023-01-03,"{""request_id"":""6490808132569"",""rows"":[{""row_id..."
6694,6511246,2023-01-03,"{""request_id"":""6511246346075"",""rows"":[{""row_id..."
6707,6500109,2023-01-03,"{""request_id"":""6500109894912"",""rows"":[{""row_id..."
6722,6499664,2023-01-03,"{""request_id"":""6499664617135"",""rows"":[{""row_id..."
...,...,...,...
36048,7280119,2023-12-08,"{""request_id"":""7280119797671"",""rows"":[{""row_id..."
36049,7304146,2023-11-15,"{""request_id"":""7304146410727"",""rows"":[{""row_id..."
36050,7281439,2023-11-27,"{""request_id"":""7281439715885"",""rows"":[{""row_id..."
36051,7373007,2023-12-13,"{""request_id"":""7373007690715"",""rows"":[{""row_id..."


#### keep latest, remove dupes

In [7]:
df.drop_duplicates(
    subset=['bigAccountId'],
    keep='last', 
    inplace=True,
)
# show
df

,bigAccountId,dtmFunded,strRequest
6690,6507952,2023-01-03,"{""request_id"":""650795220692"",""rows"":[{""row_id""..."
6691,6490808,2023-01-03,"{""request_id"":""6490808132569"",""rows"":[{""row_id..."
6694,6511246,2023-01-03,"{""request_id"":""6511246346075"",""rows"":[{""row_id..."
6707,6500109,2023-01-03,"{""request_id"":""6500109894912"",""rows"":[{""row_id..."
6722,6499664,2023-01-03,"{""request_id"":""6499664617135"",""rows"":[{""row_id..."
...,...,...,...
36048,7280119,2023-12-08,"{""request_id"":""7280119797671"",""rows"":[{""row_id..."
36049,7304146,2023-11-15,"{""request_id"":""7304146410727"",""rows"":[{""row_id..."
36050,7281439,2023-11-27,"{""request_id"":""7281439715885"",""rows"":[{""row_id..."
36051,7373007,2023-12-13,"{""request_id"":""7373007690715"",""rows"":[{""row_id..."


#### Save & upload

In [8]:
%%time

# save
str_filename = 'df_requests.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

CPU times: total: 1min 30s
Wall time: 1min 31s


In [9]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_filename}', 
    str_bucket_name=str_project,
)

CPU times: total: 16.3 s
Wall time: 34.3 s
